# Turn a new GitHub issue into an investigation

Someone opens an issue. Your agent checks out the repository, reproduces the problem in an isolated sandbox, and posts a useful investigation back to GitHub.

```mermaid
sequenceDiagram
    participant User
    participant GitHub
    participant App as Webhook receiver
    participant Agent as Agents API
    participant Sandbox
    User->>GitHub: Open a new issue
    GitHub->>App: Send a signed issues.opened webhook
    App->>Agent: Create a self-hosted session
    App->>Sandbox: Clone the repository and start the executor
    Agent->>Sandbox: Inspect code and reproduce the bug
    Agent-->>App: Root cause and suggested fix
    App-->>GitHub: Post the investigation as a comment
```


## Agents API capabilities

Sandbox, Workspace files, Streaming.

### Start from a real product event

A signed GitHub webhook triggers the investigation automatically, without asking a developer to open a separate chat or manually restate the issue.

### Give the agent a real repository

A self-hosted environment lets the agent inspect checked-out files, run the project's tests, and write a concrete investigation.

### Keep each investigation isolated

Every issue receives a fresh workspace and sandbox, which are removed when the investigation finishes.

### Return the answer to GitHub

Your application posts the agent's report back to the issue, where the people who reported the bug can act on it.


## Application flow

1. GitHub issue.
2. Signed webhook.
3. Agent session.
4. Isolated checkout.
5. Issue comment.


## What you need

- Python 3.14+ and `uv`.
- A sandbox: self-hosted Docker or a [third-party provider](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers).
- An OpenAI API key and a separate restricted executor key.
- A GitHub token and webhook secret when you connect a real repository.


## 1. Set up the investigator

From the repository root:

```bash
cp examples/agents_api/apps/github_issues/.env.example examples/agents_api/apps/github_issues/.env
docker build -t agent-api-sandbox:latest examples/agents_api/sandboxes/application_managed/docker
```

Set `OPENAI_API_KEY` and `OPENAI_EXECUTOR_API_KEY` in `examples/agents_api/apps/github_issues/.env`. Use keys with the same owner, organization, and project. Only the executor key enters the sandbox. It needs `api.agents.environments.connect` and IP restrictions that allow the sandbox's outbound network. You can also replace Docker with a compatible [sandbox provider](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers).

To create an executor key with the required permission, open [Agents > Environments > Keys](https://platform.openai.com/agents?tab=environments&environment_view=keys) and select **Create**.


## 2. Investigate the included issue

```bash
uv run examples/agents_api/apps/github_issues/main.py --issue
```

The sample repository contains a real failing shipping test. The agent should identify why express orders accidentally qualify for free shipping and explain the smallest safe fix.


## 3. Connect a real repository

Set `GITHUB_WEBHOOK_SECRET` and `GITHUB_TOKEN` in `examples/agents_api/apps/github_issues/.env`, then start the receiver. The token needs repository read access and permission to post issue comments; it also authenticates private repository clones without appearing in the Git command:

```bash
uv run examples/agents_api/apps/github_issues/main.py
```

Expose the application through your normal hosting provider. In your GitHub repository's webhook settings:

1. Set the payload URL to `https://your-app.example/webhooks/github`.
2. Select `application/json` and enter the same webhook secret.
3. Subscribe to **Issues** events.
4. Open an issue and wait for the agent's investigation comment.

The receiver verifies GitHub signatures, ignores duplicate deliveries, and releases the sandbox after each investigation. The agent is instructed to inspect without editing; the application only posts an investigation comment and never opens a pull request.

For deployment, persist delivery IDs and use a durable job queue so failed investigations can be retried after a restart.


## Run this notebook

Use a Jupyter Python kernel (Python 3.11 or later) on macOS or Linux in a local clone of the [Cookbook repository](https://github.com/openai/openai-cookbook). The application itself uses Python 3.14; `uv run` installs the dependencies declared in `main.py` and selects that interpreter.

The terminal commands above run the checked-in application. The notebook instead builds a separate copy inside an ignored `tmp_` workspace under your Cookbook checkout. Each `%%writefile` cell contains actual application source. Run these cells in order: the first cell for a module creates its file, and later cells append to it. Python definitions are executed by the application when you launch it.

The setup cell copies only the listed supporting fixtures, manifests, and policy files. It creates a fresh `.env` from the example template without copying your existing credentials. Configure the printed `.env` path before the optional launch step. Rerunning setup creates a new workspace; keep the previous workspace if you need its reports or memory.

Default execution builds and checks the files locally. Docker builds and live API calls require the explicit flags in the launch section.


In [ ]:
from pathlib import Path
import shutil
import tempfile

if globals().get("application_process") is not None and application_process.poll() is None:
    raise RuntimeError("Stop the running application before creating a new workspace.")
application_process = None

# Start Jupyter anywhere inside the Cookbook checkout.
working_directory = Path.cwd().resolve()
cookbook_root = next(
    (path for path in [working_directory, *working_directory.parents]
     if (path / "examples/agents_api/apps/github_issues/main.py").is_file()),
    None,
)
if cookbook_root is None:
    raise FileNotFoundError("Clone openai/openai-cookbook and start Jupyter inside it.")

notebook_root = Path(tempfile.mkdtemp(prefix="tmp_agents_github_issues_", dir=cookbook_root))
application_dir = notebook_root / "examples/agents_api/apps/github_issues"
application_dir.mkdir(parents=True)
for package in [notebook_root / "examples", notebook_root / "examples/agents_api",
                notebook_root / "examples/agents_api/apps", application_dir]:
    (package / "__init__.py").touch()

support_paths = [
    ".env.example",
    "sample_event.json",
    "sample_repository/shipping.py",
    "sample_repository/test_shipping.py"
]
source_dir = cookbook_root / "examples/agents_api/apps/github_issues"
for relative_path in support_paths:
    destination = application_dir / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_dir / relative_path, destination)
shutil.copy2(application_dir / ".env.example", application_dir / ".env")
shutil.copytree(
    cookbook_root / "examples/agents_api/sandboxes/application_managed/docker",
    notebook_root / "examples/agents_api/sandboxes/application_managed/docker",
)
print(f"Application workspace: {application_dir}")
print(f"Configure credentials in: {application_dir / '.env'}")


## Implementation walkthrough

This walkthrough covers GitHub webhooks, authenticated repository checkouts, isolated coding sandboxes, test execution, and issue comments.

Follow the setup instructions above, then build the application with the code cells below.


### 1. Set up the investigator and build its sandbox

Clone the repository, copy the example's environment template, and build the Docker image containing Git, Python, and the Codex executor.


### 2. Receive a GitHub issue webhook

When an issue arrives, verify its signature, skip duplicate deliveries, and start the longer investigation in the background. The runnable example implements these checks.

Always implement signature verification and delivery deduplication in production. The included application already includes both.


### 3. Check out the repository in a fresh workspace

Read the repository clone URL from the verified webhook and create a shallow checkout in a separate temporary directory for each investigation.

The runnable example authenticates private clones using Git configuration passed through the subprocess environment, not a token embedded in the URL or command.


### 4. Create an agent session for the issue

Tell the agent to reproduce the problem and explain the smallest likely fix. Ask it to write an investigation file rather than modifying source code or opening a pull request.


### 5. Attach the coding sandbox

Mount the checked-out repository into Docker and run the Codex executor with the session's environment ID. The executor connects outbound to the Agents API.


### 6. Stream the investigation and read its report

Send the issue title and body as input. The agent can inspect source files, run tests, and write a concise Markdown report in the shared workspace.


### 7. Post the findings back to the GitHub issue

Publish the generated report through the GitHub Issues API. The report appears in the original conversation, so the reporter and maintainers can immediately see the diagnosis.

Only send credentials to trusted github.com API URLs and grant the GitHub token the minimum issue-comment permissions it needs.


### 8. Release the sandbox and session

Always remove the Docker container and delete the agent session after the investigation. The temporary workspace is deleted when its surrounding context exits.


### 9. Try the sample issue or connect GitHub

The included sample reproduces a real failing shipping test without requiring GitHub credentials. Add a webhook secret and an issue-comment token when you are ready to connect a repository.

When connecting a real repository, configure its webhook to send Issues events to https://your-app.example/webhooks/github.


## Build the application

The following cells include every application module. Run all cells for each file before launching. The inline dependency declaration in `main.py` installs the current OpenAI SDK and the application libraries through `uv`.


### github.py

Clone repositories with credentials outside the command URL and post reports only to trusted GitHub API URLs.


In [ ]:
%%writefile "{application_dir}/github.py"
"""Verify GitHub deliveries, clone repositories, and post findings."""

from __future__ import annotations

import base64
import hashlib
import hmac
import os
import subprocess
from pathlib import Path
from typing import Any

import httpx


def verify_signature(payload: bytes, signature: str, secret: str) -> bool:
    expected = (
        "sha256=" + hmac.new(secret.encode(), payload, hashlib.sha256).hexdigest()
    )
    return hmac.compare_digest(signature, expected)


def clone_repository(url: str, workspace: Path) -> None:
    environment = os.environ.copy()
    if token := environment.get("GITHUB_TOKEN"):
        credentials = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        environment.update(
            {
                "GIT_CONFIG_COUNT": "1",
                "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {credentials}",
            }
        )

    subprocess.run(
        ["git", "clone", "--depth", "1", url, str(workspace)],
        check=True,
        capture_output=True,
        text=True,
        env=environment,
    )


async def post_findings(issue: dict[str, Any], findings: str) -> None:
    token = os.environ.get("GITHUB_TOKEN")
    comments_url = str(issue.get("comments_url", ""))
    if token and comments_url.startswith("https://api.github.com/"):
        async with httpx.AsyncClient() as http:
            response = await http.post(
                comments_url,
                json={"body": findings},
                headers={
                    "Authorization": f"Bearer {token}",
                    "Accept": "application/vnd.github+json",
                },
            )
            response.raise_for_status()
    else:
        print(findings)


### agent.py

Investigate an issue in a self-hosted sandbox, stream the answer, read the report, and release the session and container.


In [ ]:
%%writefile "{application_dir}/agent.py"
"""Investigate a GitHub issue in an isolated Agents API sandbox."""

from __future__ import annotations

import asyncio
import os
import sys
from pathlib import Path
from typing import Any

import docker
from docker.models.containers import Container
from openai import AsyncOpenAI, NotFoundError

INSTRUCTIONS = """\
Investigate GitHub issues by inspecting the repository and reproducing problems when practical.
Identify the likely cause and recommend the smallest fix.
Write your findings to /workspace/investigation.md.
Do not edit source files or contact external services.
"""


def start_executor(workspace: Path, environment_id: str, remote_url: str) -> Container:
    return docker.from_env().containers.run(
        os.environ.get("AGENTS_SANDBOX_IMAGE", "agent-api-sandbox:latest"),
        [
            "codex",
            "exec-server",
            "--remote",
            remote_url,
            "--environment-id",
            environment_id,
        ],
        environment={"CODEX_API_KEY": os.environ["OPENAI_EXECUTOR_API_KEY"]},
        volumes={str(workspace.resolve()): {"bind": "/workspace", "mode": "rw"}},
        detach=True,
        auto_remove=True,
    )


async def investigate_issue(issue: dict[str, Any], workspace: Path) -> dict[str, Any]:
    container: Container | None = None
    async with AsyncOpenAI() as client:
        session = await client.beta.agents.sessions.create(
            agent={
                "model": os.environ.get("OPENAI_MODEL", "gpt-5.6-sol"),
                "instructions": INSTRUCTIONS,
                "reasoning": {"effort": "high"},
            },
            environment={"type": "self_hosted", "workspace_directory": "/workspace"},
        )

        try:
            environment = session.environment
            if environment.type != "self_hosted":
                raise RuntimeError("Expected a self-hosted execution environment.")
            container = await asyncio.to_thread(
                start_executor, workspace, environment.id, environment.remote_url
            )
            prompt = f"""\
Investigate GitHub issue #{issue.get("number", "?")}: {issue["title"]}

{issue.get("body", "")}

Inspect /workspace, run relevant tests, identify the root cause, and write
/workspace/investigation.md with your findings and a proposed fix.
"""
            parts: list[str] = []
            async with client.beta.agents.sessions.stream(
                session.id, input=prompt
            ) as events:
                async for event in events:
                    if event.type == "agent.session.turn.output_text.delta":
                        parts.append(event.delta)
                    elif (
                        event.type == "agent.session.turn.output_text.done"
                        and not parts
                    ):
                        parts.append(event.text)
                    if event.type in {
                        "agent.session.failed",
                        "agent.session.turn.failed",
                        "error",
                    }:
                        raise RuntimeError(
                            f"Issue investigation failed: {event.to_dict()}"
                        )
                    if event.type == "agent.session.turn.cancelled":
                        raise RuntimeError("Issue investigation was cancelled.")

            report = workspace / "investigation.md"
            if not report.exists():
                raise RuntimeError(
                    "The agent did not create /workspace/investigation.md."
                )
            return {
                "issue_number": issue.get("number"),
                "session_id": session.id,
                "summary": "".join(parts),
                "findings": report.read_text(),
            }
        finally:
            original_error = sys.exception()
            cleanup_errors: list[Exception] = []
            try:
                if container is not None:
                    await asyncio.to_thread(container.remove, force=True)
            except docker.errors.NotFound:
                pass
            except Exception as error:
                cleanup_errors.append(error)
            try:
                await client.beta.agents.sessions.delete(session.id)
            except NotFoundError:
                pass
            except Exception as error:
                cleanup_errors.append(error)
            if original_error is not None:
                for error in cleanup_errors:
                    original_error.add_note(
                        f"Cleanup for session {session.id}: {error}"
                    )
            elif cleanup_errors:
                raise ExceptionGroup(
                    f"Could not clean up session {session.id} and its sandbox",
                    cleanup_errors,
                )


### main.py

Verify and deduplicate GitHub webhooks, queue investigations, or run the bundled failing-test example.


In [ ]:
%%writefile "{application_dir}/main.py"
# /// script
# requires-python = ">=3.14"
# dependencies = [
#     "openai>=3.13.0",
#     "docker",
#     "fastapi",
#     "python-dotenv",
#     "uvicorn",
# ]
# ///

"""Receive GitHub issue webhooks or investigate the included issue."""

from __future__ import annotations

import argparse
import asyncio
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from fastapi import BackgroundTasks, FastAPI, HTTPException, Request

# Support direct execution from any working directory.
if __package__ in {None, ""}:
    sys.path.insert(0, str(Path(__file__).resolve().parents[4]))


from examples.agents_api.apps.github_issues.agent import investigate_issue
from examples.agents_api.apps.github_issues.github import (
    clone_repository,
    post_findings,
    verify_signature,
)

EXAMPLE_DIR = Path(__file__).resolve().parent
DELIVERIES: set[str] = set()


async def handle_github_event(event: dict[str, Any]) -> None:
    issue = event["issue"]
    clone_url = event["repository"]["clone_url"]
    with tempfile.TemporaryDirectory(
        prefix=".agent-github-", dir=EXAMPLE_DIR
    ) as directory:
        workspace = Path(directory) / "repository"
        await asyncio.to_thread(clone_repository, clone_url, workspace)
        result = await investigate_issue(issue, workspace)

    await post_findings(issue, result["findings"])


app = FastAPI(title="GitHub issue investigator")


@app.post("/webhooks/github")
async def github_webhook(request: Request, tasks: BackgroundTasks) -> dict[str, str]:
    body = await request.body()
    secret = os.environ.get("GITHUB_WEBHOOK_SECRET", "")
    signature = request.headers.get("x-hub-signature-256", "")
    if not secret or not verify_signature(body, signature, secret):
        raise HTTPException(status_code=401, detail="Invalid webhook signature.")

    delivery = request.headers.get("x-github-delivery", "")
    if delivery and delivery in DELIVERIES:
        return {"status": "already_processed"}

    event = json.loads(body)
    event_name = request.headers.get("x-github-event", "")
    if event_name != "issues" or event.get("action") not in {
        "opened",
        "edited",
        "reopened",
    }:
        return {"status": "ignored"}

    if delivery:
        DELIVERIES.add(delivery)
    tasks.add_task(handle_github_event, event)
    return {"status": "accepted"}


async def investigate_sample_issue() -> None:
    event = json.loads((EXAMPLE_DIR / "sample_event.json").read_text())
    with tempfile.TemporaryDirectory(
        prefix=".agent-github-", dir=EXAMPLE_DIR
    ) as directory:
        workspace = Path(directory) / "repository"
        shutil.copytree(
            EXAMPLE_DIR / "sample_repository",
            workspace,
            ignore=shutil.ignore_patterns("__pycache__"),
        )
        result = await investigate_issue(event["issue"], workspace)
        print(result["findings"])




Continue `main.py`: `main`, `Application entry point`. This cell appends to the same file.


In [ ]:
%%writefile -a "{application_dir}/main.py"
def main() -> None:
    parser = argparse.ArgumentParser(
        description="Investigate new GitHub issues with Agents API."
    )
    parser.add_argument(
        "--issue", action="store_true", help="Investigate the included issue."
    )
    args = parser.parse_args()
    load_dotenv(EXAMPLE_DIR / ".env")
    if args.issue:
        asyncio.run(investigate_sample_issue())
        return

    import uvicorn

    uvicorn.run(app, host="127.0.0.1", port=8002)


if __name__ == "__main__":
    main()


## Check the generated files

Compile all generated modules without importing them or contacting external services. This catches syntax errors before you launch the application.


In [ ]:
import py_compile

modules = ["github.py", "agent.py", "main.py"]
for filename in modules:
    py_compile.compile(str(application_dir / filename), doraise=True)
print(f"Compiled {len(modules)} application modules.")


## Launch the application (optional)

Edit the generated `.env` file with the credentials listed above. Install `uv` and, for sandbox applications, start Docker. The following cells are disabled by default. Enabling them may incur API usage and connect to the configured services.

Use the generated workspace for every path below. For a hosted deployment, package the generated application files and supply credentials through your deployment's secret configuration.


In [ ]:
import subprocess

BUILD_SANDBOX = False
if BUILD_SANDBOX:
    subprocess.run(
        ["docker", "build", "-t", "agent-api-sandbox:latest", str(notebook_root / "examples/agents_api/sandboxes/application_managed/docker")],
        cwd=notebook_root,
        check=True,
    )


The launch arguments investigate the bundled issue in the local sample repository without posting a GitHub comment. Set `application_arguments = []` to run the persistent webhook receiver instead. The process writes to `application.log` in the generated workspace. Inspect that file for errors and progress.


In [ ]:
import os
import subprocess

RUN_APPLICATION = False
application_arguments = ["--issue"]
if RUN_APPLICATION:
    if application_process is not None and application_process.poll() is None:
        raise RuntimeError("Stop the previous application before launching again.")
    with (application_dir / "application.log").open("w") as application_log:
        application_process = subprocess.Popen(
            ["uv", "run", str(application_dir / "main.py"), *application_arguments],
            cwd=notebook_root,
            env={key: value for key, value in os.environ.items() if key != "VIRTUAL_ENV"},
            start_new_session=True,
            stdout=application_log,
            stderr=subprocess.STDOUT,
        )
    print(f"Process started: {application_process.pid}")
    print(f"Progress log: {application_dir / 'application.log'}")


### Stop a running application

Set `STOP_APPLICATION = True` after you finish. Interrupt the application process group so the application's shutdown handlers can close sessions and remove containers. Batch commands normally exit on their own. A timeout means shutdown is still in progress; inspect the log before taking further action.


In [ ]:
import os
import signal

STOP_APPLICATION = False
if STOP_APPLICATION and application_process is not None:
    if application_process.poll() is None:
        os.killpg(os.getpgid(application_process.pid), signal.SIGINT)
        application_process.wait(timeout=30)
    print(f"Application exited with status {application_process.returncode}.")


The generated workspace remains available for reports and memory. Remove it manually after stopping the application and saving any files you need. Do not rerun the launch cell while the previous process is running.


## Example result

A real issue produces a test-backed investigation in the same place your team already tracks the bug.

The following illustrates a possible result; model-generated findings depend on the inputs and connected sources.

```text
Issue #42: Express shipping becomes free for orders over $100

Reproduction:
shipping_cost(125, express=True) returned 0; expected 15.

Root cause:
The free-shipping condition runs before the express-shipping check.

Suggested fix:
Check express shipping first, then apply the standard-order discount.
```


## Next steps

- Replace local Docker with an isolated hosted sandbox provider when deploying the webhook receiver.
- Persist delivery IDs and session references so retries remain safe across application restarts.
- Add an explicit human approval step before applying code changes or opening a pull request.


## Related documentation

- [Sandbox providers](https://developers.openai.com/api/docs/guides/agents-api/environments/self-hosted#sandbox-providers): Choose a local or hosted sandbox for isolated repository investigations.


## Files

- [main.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/github_issues/main.py): Webhook handling and the included-issue entrypoint.
- [agent.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/github_issues/agent.py): Sandbox investigation and report collection.
- [github.py](https://github.com/openai/openai-cookbook/blob/main/examples/agents_api/apps/github_issues/github.py): Signature verification, repository cloning, and comments.
